# Refactored App Logic

**Doc ID**: `notebook.refactored-app-logic`  
**Status**: `working`  
**Kind**: `refactor-notebook`  
**Last Updated**: `2026-04-04`

This notebook implements the refactored agentic search pipeline with:
- Source-type-routed multi-entity extraction (roundup pages yield N rows)
- Supervisor ReAct loop (observe table state, decide next action, act)
- Fuzzy entity deduplication (normalized name + Jaccard token overlap)
- Confidence-weighted ranking (identity columns 3x, enrichment 1x)
- Rewritten prompts with system/user separation and grounding rules

Run each cell in order. Cells 1-9 define standalone functions; Cell 10 wires them into the full pipeline.

In [1]:
import os, json, re, time, csv, io
from dataclasses import dataclass, field
from enum import Enum
from urllib.parse import urlparse, urlunparse
import requests
from html import unescape

# ── API Keys ──────────────────────────────────────────────────────
# Load from .env.local at repo root
_repo_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
_env_path = os.path.join(_repo_root, ".env.local")
if not os.path.exists(_env_path):
    _env_path = os.path.join(os.getcwd(), "..", "..", ".env.local")
if os.path.exists(_env_path):
    for line in open(_env_path):
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())

BRAVE_API_KEY = os.environ.get("BRAVE_API_KEY", "")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
JINA_API_KEY = os.environ.get("JINA_API_KEY", "jina_bfbbca8d9d304940924bf55012661bd3khvJ4acUMcNdfgKBitILcI3rI47c")

assert BRAVE_API_KEY and BRAVE_API_KEY != "replace_me", "Set BRAVE_API_KEY in .env.local"
assert GEMINI_API_KEY and GEMINI_API_KEY != "replace_me", "Set GEMINI_API_KEY in .env.local"

# ── Model Tiers ───────────────────────────────────────────────────
# Split by task: 3.x preview for reasoning-heavy, 2.5 lite for reformatting
#
# Rationale (SimpleQA / GPQA benchmarks):
#   2.5 Flash-Lite: 11.5% SimpleQA — only suited for pure reformatting
#   3.1 Flash-Lite: 43.3% SimpleQA, 86.9% GPQA — 4x better factual grounding
#   3 Flash:        90.4% GPQA — strongest non-pro reasoning
#
# Extractor is split: roundup pages need multi-entity reasoning (3 Flash),
# single entity pages are simpler (3.1 Flash-Lite at lower cost).
MODELS = {
    "planner":           "gemini-3.1-flash-lite-preview",  # needs real factual reasoning for schema design
    "extractor_roundup": "gemini-3-flash-preview",         # multi-entity extraction from listicles
    "extractor_entity":  "gemini-3.1-flash-lite-preview",  # single-entity pages, simpler task
    "supervisor":        "gemini-3-flash-preview",         # observe/decide loop, multi-step reasoning
    "verifier":          "gemini-3.1-flash-lite-preview",  # factual verification needs grounding
    "rewriter":          "gemini-2.5-flash-lite",          # pure reformatting, no factual recall needed
}

# Ordered fallbacks when preview models 429 (same key shares quota; extra tier helps).
MODEL_FALLBACK_CHAIN = {
    "gemini-3-flash-preview": ["gemini-2.5-flash", "gemini-2.5-flash-lite"],
    "gemini-3.1-flash-lite-preview": ["gemini-2.5-flash-lite"],
}

_last_gemini_call_ts = 0.0
GEMINI_MIN_INTERVAL = 2.0  # seconds between calls to avoid RPM burst

# ── Gemini Client ─────────────────────────────────────────────────
def _gemini_post(model: str, body: dict) -> dict:
    """Single Gemini API call with throttle. Returns parsed JSON response."""
    global _last_gemini_call_ts
    elapsed = time.time() - _last_gemini_call_ts
    if elapsed < GEMINI_MIN_INTERVAL:
        time.sleep(GEMINI_MIN_INTERVAL - elapsed)

    url = (f"https://generativelanguage.googleapis.com/v1beta/models/"
           f"{model}:generateContent?key={GEMINI_API_KEY}")
    resp = requests.post(url, json=body, timeout=30)
    _last_gemini_call_ts = time.time()
    return resp


def gemini_call(operation: str, system: str, user: str,
                temperature: float = 0.2, json_mode: bool = True):
    """Call Gemini with system/user message separation. Returns parsed JSON or raw text."""
    primary_model = MODELS.get(operation, "gemini-2.5-flash-lite")
    chain = MODEL_FALLBACK_CHAIN.get(primary_model, [])
    models_to_try: list[str] = []
    for m in [primary_model, *chain]:
        if m not in models_to_try:
            models_to_try.append(m)

    body = {
        "systemInstruction": {"parts": [{"text": system}]},
        "contents": [{"role": "user", "parts": [{"text": user}]}],
        "generationConfig": {"temperature": temperature},
    }
    if json_mode:
        body["generationConfig"]["responseMimeType"] = "application/json"

    last_error: Exception | None = None
    payload = None
    model_used = primary_model

    for model in models_to_try:
        payload = None
        for attempt in range(6):
            try:
                resp = _gemini_post(model, body)
                if resp.status_code == 429:
                    last_error = Exception(f"429 on {model}")
                    time.sleep(min(90, 8 + attempt * 14))
                    continue
                if resp.status_code == 503:
                    last_error = Exception(f"503 on {model}")
                    time.sleep(4 + attempt * 2)
                    continue
                resp.raise_for_status()
                payload = resp.json()
                model_used = model
                break
            except requests.exceptions.ReadTimeout as e:
                last_error = e
                time.sleep(min(45, 4 + attempt * 8))
        if payload is not None:
            if model_used != primary_model:
                print(f"    (used fallback {model_used} for {operation})")
            break

    if payload is None:
        raise RuntimeError(f"Gemini failed on all models ({operation}): {last_error}")

    candidates = payload.get("candidates") or []
    first = candidates[0] if candidates else {}
    parts = first.get("content", {}).get("parts") or []

    text = ""
    for part in parts:
        text += part.get("text", "")
    text = text.strip()

    if json_mode:
        fenced = re.search(r"```json\s*([\s\S]*?)```", text, re.IGNORECASE)
        json_text = fenced.group(1).strip() if fenced else text
        if not json_text:
            raise ValueError(f"Gemini returned empty JSON payload ({operation}, {model_used})")
        parsed = json.loads(json_text)
        if parsed is None:
            raise ValueError(f"Gemini returned null ({operation}, {model_used})")
        return parsed
    return text

# ── Shared Types ──────────────────────────────────────────────────
class SourceClass(Enum):
    ROUNDUP = "roundup"
    ENTITY_PAGE = "entity_page"
    OFFICIAL_SITE = "official_site"
    DIRECTORY = "directory"
    FORUM = "forum"

@dataclass
class SearchResult:
    title: str
    url: str
    description: str
    age: str = ""

@dataclass
class ParsedDocument:
    url: str
    final_url: str
    title: str
    description: str
    text: str
    source_class: SourceClass = SourceClass.ENTITY_PAGE
    fetched_at: float = field(default_factory=time.time)

@dataclass
class CellValue:
    key: str
    value_text: str | None
    state: str            # filled | not_found | unsupported | uncertain | conflict
    confidence: float
    evidence_snippet: str = ""

@dataclass
class CriterionVerdict:
    label: str
    verdict: str          # pass | fail | uncertain | conflict
    summary: str
    confidence: float
    evidence_snippet: str = ""

@dataclass
class ExtractedRow:
    canonical_name: str
    canonical_url: str
    cells: list[CellValue]
    criteria_verdicts: list[CriterionVerdict]
    extraction_confidence: float
    source_url: str
    source_class: SourceClass = SourceClass.ENTITY_PAGE

@dataclass
class MergedRow:
    canonical_name: str
    canonical_url: str
    cells: list[CellValue]
    criteria_verdicts: list[CriterionVerdict]
    score: float
    source_urls: list[str] = field(default_factory=list)
    status: str = "uncertain"

@dataclass
class VerifiedRow(MergedRow):
    verification_summary: str = ""

@dataclass
class RankedRow(VerifiedRow):
    rank: int = 0
    final_score: float = 0.0

@dataclass
class ColumnSpec:
    key: str
    label: str
    kind: str             # identity | criterion_summary | enrichment
    value_type: str = "string"

@dataclass
class Criterion:
    label: str
    kind: str             # hard_filter | soft_signal

@dataclass
class ResearchPlan:
    entity_type: str
    criteria: list[Criterion]
    columns: list[ColumnSpec]
    search_queries: list[str]
    notes: str = ""

@dataclass
class ColumnFillSummary:
    key: str
    label: str
    fill_rate: float      # 0.0 to 1.0
    avg_confidence: float

@dataclass
class SupervisorState:
    total_rows: int
    target_rows: int
    column_summaries: list[ColumnFillSummary]
    unfetched_urls: list[str]
    iteration: int
    max_iterations: int = 3

@dataclass
class SupervisorAction:
    action: str           # search_more | fetch_more | extract_from_existing | done
    queries: list[str] = field(default_factory=list)
    urls: list[str] = field(default_factory=list)
    focus_columns: list[str] = field(default_factory=list)
    reasoning: str = ""

# ── Global State ──────────────────────────────────────────────────
document_store: dict[str, ParsedDocument] = {}
all_search_results: list[SearchResult] = []

# ── Test Queries ──────────────────────────────────────────────────
TEST_QUERIES = [
    ("best pizza places in Brooklyn", 10),
    ("YC W24 healthcare startups", 15),
    ("open source LLM projects with >1k stars", 12),
]

print("Setup complete.")
print(f"  Brave API key: ...{BRAVE_API_KEY[-4:]}")
print(f"  Gemini API key: ...{GEMINI_API_KEY[-4:]}")
print(f"\n  Model map:")
for task, model in MODELS.items():
    print(f"    {task:20s} -> {model}")

Setup complete.
  Brave API key: ...hf5t
  Gemini API key: ...A4lw

  Model map:
    planner              -> gemini-3.1-flash-lite-preview
    extractor_roundup    -> gemini-3-flash-preview
    extractor_entity     -> gemini-3.1-flash-lite-preview
    supervisor           -> gemini-3-flash-preview
    verifier             -> gemini-3.1-flash-lite-preview
    rewriter             -> gemini-2.5-flash-lite


## Cell 1 — Query Planning

Rewritten planner prompt with system/user separation and one few-shot example. Uses `gemini-2.5-flash-lite`.

In [2]:
PLANNER_SYSTEM = """You are a research planner for grounded entity discovery. Your job is to analyze a natural-language research query and produce a structured plan that a pipeline of search + extraction + verification agents will execute.

You must return:
- entity_type: the kind of entity the user is looking for
- criteria: hard_filter (must match) and soft_signal (nice to have) rules that determine inclusion
- columns: data fields to extract per entity, with types — always include identity columns (name, url) and relevant enrichment
- search_queries: 3–6 diverse web search queries designed to surface pages containing these entities
- notes: brief planning rationale

Design columns like a human researcher would build a spreadsheet.

Example — for "best Italian restaurants in Manhattan", a good plan:
{
  "entity_type": "business",
  "criteria": [
    {"label": "Located in Manhattan", "kind": "hard_filter"},
    {"label": "Serves Italian cuisine", "kind": "hard_filter"},
    {"label": "Well-reviewed or critically acclaimed", "kind": "soft_signal"}
  ],
  "columns": [
    {"key": "name",           "label": "Name",           "kind": "identity",   "value_type": "string"},
    {"key": "website",        "label": "Website",        "kind": "identity",   "value_type": "url"},
    {"key": "address",        "label": "Address",        "kind": "enrichment", "value_type": "string"},
    {"key": "neighborhood",   "label": "Neighborhood",   "kind": "enrichment", "value_type": "string"},
    {"key": "cuisine_style",  "label": "Style",          "kind": "enrichment", "value_type": "string"},
    {"key": "price_range",    "label": "Price Range",    "kind": "enrichment", "value_type": "string"},
    {"key": "rating",         "label": "Rating",         "kind": "enrichment", "value_type": "number"},
    {"key": "notable_dishes", "label": "Notable Dishes", "kind": "enrichment", "value_type": "string"}
  ],
  "search_queries": [
    "best Italian restaurants Manhattan 2024",
    "top rated Italian Manhattan NYC",
    "Michelin Italian restaurant Manhattan",
    "Eater best Italian Manhattan"
  ],
  "notes": "Targeting sit-down Italian restaurants in Manhattan with editorial and review coverage."
}"""

PLANNER_USER = """Query: {query}
Target results: {target_results}

Return a JSON object with keys: entity_type, criteria, columns, search_queries, notes."""


def plan_query(query: str, target_results: int = 10) -> ResearchPlan:
    result = gemini_call(
        "planner",
        system=PLANNER_SYSTEM,
        user=PLANNER_USER.format(query=query, target_results=target_results),
    )
    return ResearchPlan(
        entity_type=result.get("entity_type", "unknown"),
        criteria=[
            Criterion(label=c["label"], kind=c.get("kind", "hard_filter"))
            for c in result.get("criteria", [])
        ],
        columns=[
            ColumnSpec(
                key=c["key"], label=c["label"],
                kind=c.get("kind", "enrichment"),
                value_type=c.get("value_type", "string"),
            )
            for c in result.get("columns", [])
        ],
        search_queries=result.get("search_queries", [query])[:6],
        notes=result.get("notes", ""),
    )


# ── Test on first query ──────────────────────────────────────────
query, target = TEST_QUERIES[0]
plan = plan_query(query, target)

print(f"Entity type: {plan.entity_type}")
print(f"\nCriteria ({len(plan.criteria)}):")
for c in plan.criteria:
    print(f"  [{c.kind}] {c.label}")
print(f"\nColumns ({len(plan.columns)}):")
for c in plan.columns:
    print(f"  {c.key} ({c.kind}, {c.value_type}): {c.label}")
print(f"\nSearch queries ({len(plan.search_queries)}):")
for q in plan.search_queries:
    print(f"  {q}")
print(f"\nNotes: {plan.notes}")

Entity type: business

Criteria (3):
  [hard_filter] Located in Brooklyn, NY
  [hard_filter] Primary business is pizza
  [soft_signal] High ratings or critical acclaim

Columns (8):
  name (identity, string): Name
  url (identity, url): Website
  address (enrichment, string): Address
  neighborhood (enrichment, string): Neighborhood
  pizza_style (enrichment, string): Style (e.g., Neapolitan, NY Slice)
  price_range (enrichment, string): Price Range
  rating (enrichment, number): Rating
  signature_pie (enrichment, string): Signature Pie

Search queries (5):
  best pizza in Brooklyn 2024
  top rated pizza spots Brooklyn NYC
  must try pizza Brooklyn guide
  best pizza Brooklyn Eater
  iconic pizza places Brooklyn

Notes: Focusing on highly-regarded pizza establishments in Brooklyn, prioritizing those with distinct styles and strong editorial presence.


## Cell 2 — Brave Search

Real Brave API calls with URL-level dedup across all planned queries.

In [5]:
def normalize_url(url: str) -> str:
    parsed = urlparse(url.lower().rstrip("/"))
    path = re.sub(r"/+", "/", parsed.path).rstrip("/")
    return f"{parsed.netloc}{path}"


def search_brave(queries: list[str], results_per_query: int = 10) -> list[SearchResult]:
    seen: set[str] = set()
    results: list[SearchResult] = []

    for q in queries:
        try:
            resp = requests.get(
                "https://api.search.brave.com/res/v1/web/search",
                params={"q": q, "count": results_per_query,
                        "text_decorations": "false", "spellcheck": "false"},
                headers={"X-Subscription-Token": BRAVE_API_KEY,
                         "Accept": "application/json"},
                timeout=15,
            )
            resp.raise_for_status()
        except Exception as e:
            print(f"  Search failed for '{q}': {e}")
            continue

        for r in resp.json().get("web", {}).get("results", []):
            url = r.get("url", "")
            title = r.get("title", "")
            if not url or not title:
                continue
            norm = normalize_url(url)
            if norm in seen:
                continue
            seen.add(norm)
            results.append(SearchResult(
                title=title, url=url,
                description=r.get("description", ""),
                age=r.get("age", ""),
            ))

    return results


# ── Test ──────────────────────────────────────────────────────────
all_search_results = search_brave(plan.search_queries, results_per_query=10)

print(f"Found {len(all_search_results)} unique results across {len(plan.search_queries)} queries\n")
for i, r in enumerate(all_search_results[:20]):
    print(f"  {i+1:2d}. {r.title[:80]}")
    print(f"      {r.url}")
    print()

Found 19 unique results across 5 queries

   1. The Nine Best Whole-Pie Pizza Spots in Brooklyn Right Now
      https://www.bkmag.com/2026/03/02/best-whole-pie-pizza-restaurants-brooklyn/

   2. Dough Masters: The 16 Best Pizza Restaurants in Brooklyn | Best Pizza In Brookly
      https://www.foodieflashpacker.com/pizza-restaurants-in-brooklyn/

   3. The 20 Best Pizza Places In Brooklyn - New York - The Infatuation
      https://www.theinfatuation.com/new-york/guides/a-guide-to-the-best-brooklyn-pizza

   4. THE 10 BEST Pizza Places in Brooklyn (Updated 2026) - Tripadvisor
      https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html

   5. 12 Best Pizza Joints in Brooklyn (Ranked by Locals) (2026 Guide)
      https://newyorkspork.com/best-pizza-brooklyn/

   6. Brooklyn Bites: The Best Slice Shops in The Borough
      https://www.bkmag.com/2024/08/27/the-bk-five-brooklyns-best-slice-shops/

   7. The 18 Best Pizza Spots in Brooklyn - PureWow
      https://www.purewo

## Cell 3 — Fetch, Parse, and Source Type Classification

Fetches HTML, strips to plain text, and classifies each document as `roundup`, `entity_page`, `official_site`, `directory`, or `forum` using URL patterns and HTML heuristics. Documents are stored in `document_store` for re-use by the supervisor loop.

In [6]:
def _fetch_jina(url: str, char_limit: int) -> ParsedDocument | None:
    try:
        resp = requests.get(
            f"https://r.jina.ai/{url}",
            headers={
                "Authorization": f"Bearer {JINA_API_KEY}",
                "Accept": "application/json",
                "X-Return-Format": "markdown",
            },
            timeout=15,
            allow_redirects=True,
        )
        resp.raise_for_status()
        payload = resp.json()
        data = payload.get("data", {}) if isinstance(payload, dict) else {}

        text = data.get("content") if isinstance(data, dict) else None
        if not text:
            return None

        return ParsedDocument(
            url=url,
            final_url=str(data.get("url", url)),
            title=str(data.get("title", url)),
            description=str(data.get("description", "")),
            text=text[:char_limit],
        )
    except Exception as e:
        print(f"    Jina failed: {e}")
        return None


def _fetch_raw(url: str, char_limit: int) -> ParsedDocument | None:
    try:
        resp = requests.get(
            url,
            headers={
                "User-Agent": "Mozilla/5.0 (compatible; AgenticSearch/1.0)",
                "Accept": "text/html,application/xhtml+xml,*/*;q=0.8",
            },
            timeout=12,
            allow_redirects=True,
        )
        resp.raise_for_status()
        html = resp.text[: char_limit * 2]

        title_match = re.search(r"<title[^>]*>(.*?)</title>", html, re.IGNORECASE | re.DOTALL)
        title = title_match.group(1).strip() if title_match else url

        text = re.sub(r"<(script|style|noscript|svg|head)[^>]*>[\s\S]*?</\1>", " ", html, flags=re.IGNORECASE)
        text = re.sub(r"<[^>]+>", " ", text)
        text = unescape(text)
        text = re.sub(r"\s+", " ", text).strip()

        return ParsedDocument(
            url=url,
            final_url=str(resp.url),
            title=title or url,
            description="",
            text=text[:char_limit],
        )
    except Exception as e:
        print(f"    Raw failed: {url}  ({e})")
        return None


def _fetch_reddit_json(url: str, char_limit: int) -> ParsedDocument | None:
    try:
        json_url = url.rstrip("/") + ".json"
        resp = requests.get(
            json_url,
            headers={"User-Agent": "AgenticSearch/1.0"},
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()

        post = data[0]["data"]["children"][0]["data"]
        title = post.get("title", "")
        lines = [title, post.get("selftext", "")]
        for comment in data[1]["data"]["children"][:20]:
            comment_data = comment.get("data", {})
            body = comment_data.get("body", "")
            if body:
                lines.append(body)

        text = "\n".join(lines).strip()[:char_limit]
        if not text:
            return None

        return ParsedDocument(
            url=url,
            final_url=url,
            title=title or url,
            description="",
            text=text,
            source_class=SourceClass.FORUM,
        )
    except Exception as e:
        print(f"    Reddit JSON failed: {url}  ({e})")
        return None


def _is_interstitial(doc: ParsedDocument) -> bool:
    hay = f"{doc.title}\n{doc.text[:1200]}".lower()
    patterns = [
        "please wait for verification",
        "checking if the site connection is secure",
        "enable javascript and cookies to continue",
        "attention required!",
        "cloudflare",
    ]
    return any(p in hay for p in patterns)


def fetch_and_parse(url: str, char_limit: int = 12000) -> ParsedDocument | None:
    if "reddit.com/" in url and not url.endswith(".json"):
        doc = _fetch_reddit_json(url, char_limit)
        if doc and not _is_interstitial(doc):
            return doc

    doc = _fetch_jina(url, char_limit)
    if doc and len(doc.text) >= 100 and not _is_interstitial(doc):
        return doc

    fallback = _fetch_raw(url, char_limit)
    if fallback and not _is_interstitial(fallback):
        return fallback

    print(f"  Failed: {url}  (all fetch strategies failed/interstitial)")
    return None


# ── Source Type Classifier ────────────────────────────────────────
ROUNDUP_DOMAINS = {
    "eater.com", "thrillist.com", "timeout.com", "infatuation.com",
    "nymag.com", "bonappetit.com", "seriouseats.com", "tasteatlas.com",
    "techcrunch.com", "producthunt.com",
    "bkmag.com", "tastingtable.com", "purewow.com", "thedailymeal.com",
    "cntraveler.com", "foodandwine.com",
}
DIRECTORY_DOMAINS = {
    "yelp.com", "tripadvisor.com", "foursquare.com", "zomato.com",
    "opentable.com", "crunchbase.com",
}
FORUM_DOMAINS = {
    "reddit.com", "quora.com", "stackexchange.com",
    "news.ycombinator.com", "stackoverflow.com",
}


def classify_source(doc: ParsedDocument, entity_type: str = "") -> SourceClass:
    domain = urlparse(doc.final_url).netloc.lower().replace("www.", "")
    path = urlparse(doc.final_url).path.lower()

    if any(d in domain for d in FORUM_DOMAINS):
        return SourceClass.FORUM
    if any(d in domain for d in DIRECTORY_DOMAINS):
        return SourceClass.DIRECTORY
    if any(d in domain for d in ROUNDUP_DOMAINS):
        return SourceClass.ROUNDUP

    title_lower = doc.title.lower()
    text_lower = doc.text[:2000].lower()

    # GitHub list/topic pages often contain many projects; treat them as roundup sources.
    if "github.com" in domain:
        github_listy = (
            "/topics/" in path
            or "awesome" in path
            or "awesome" in title_lower
            or "top " in title_lower
            or "best " in title_lower
            or "leaderboard" in title_lower
            or "open source llm" in title_lower
        )
        if github_listy:
            return SourceClass.ROUNDUP

    has_list_title = bool(re.search(
        r"\b(\d+\s+\w*\s*(?:best|top|greatest)|best\s+\d+|top\s+\d+)\b",
        title_lower,
    )) or bool(re.search(
        r"\bbest\b.*\b(?:spots?|places?|shops?|restaurants?|joints?|projects?|startups?|companies|repos?|repositories|llms?)\b",
        title_lower,
    ))
    has_list_body = len(re.findall(
        r"(?:^|\n)\s*\d+[\.\)]\s+", doc.text[:4000])) >= 4
    has_many_links = len(re.findall(r"https?://github\.com/[^\s\)\]\"']+", text_lower)) >= 6

    if has_list_title or has_list_body or has_many_links:
        return SourceClass.ROUNDUP

    return SourceClass.ENTITY_PAGE


# ── Fetch and classify top results ────────────────────────────────
MAX_FETCH = 6
for r in all_search_results[:MAX_FETCH]:
    if r.url in document_store:
        continue
    doc = fetch_and_parse(r.url)
    if doc and len(doc.text) < 100:
        print(f"  SKIP (too short): {doc.title[:50]} ({len(doc.text)} chars)")
        continue
    if doc:
        doc.source_class = classify_source(doc, plan.entity_type)
        document_store[r.url] = doc
        print(f"  [{doc.source_class.value:14s}] {doc.title[:70]}")
        print(f"    {len(doc.text):,} chars | {doc.url}")
        print()

print(f"Fetched {len(document_store)} documents")
class_counts: dict[str, int] = {}
for d in document_store.values():
    class_counts[d.source_class.value] = class_counts.get(d.source_class.value, 0) + 1
print(f"Source types: {json.dumps(class_counts)}")

  [roundup       ] The Best Whole-Pie Pizza Spots in Brooklyn Right Now
    12,000 chars | https://www.bkmag.com/2026/03/02/best-whole-pie-pizza-restaurants-brooklyn/

  [roundup       ] Dough Masters: The 16 Best Pizza Restaurants in Brooklyn | Best Pizza 
    12,000 chars | https://www.foodieflashpacker.com/pizza-restaurants-in-brooklyn/

  [roundup       ] The 20 Best Pizza Places In Brooklyn
    12,000 chars | https://www.theinfatuation.com/new-york/guides/a-guide-to-the-best-brooklyn-pizza

    Raw failed: https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html  (403 Client Error: Forbidden for url: https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html)
  Failed: https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html  (all fetch strategies failed)
  [roundup       ] 12 Best Pizza Joints in Brooklyn (Ranked by Locals)
    12,000 chars | https://newyorkspork.com/best-pizza-brooklyn/

  [roundup       ] Brooklyn Bites: The Best

## Cell 4 — Source-Type-Routed Multi-Entity Extraction

Three extractor sub-prompts routed by `SourceClass`:
- **roundup / directory** → extract ALL entities (a "10 best" article yields 10 rows)
- **entity_page / official_site** → extract ONE entity with maximum detail
- **forum** → extract mentioned entities with low confidence

Every cell value must cite a direct snippet from the source text.

In [ ]:
EXTRACTOR_SYSTEM_ROUNDUP = """\
You are extracting entities from a web page that lists, ranks, or reviews multiple items.

Rules:
- Extract ALL distinct entities that match the research query. A "10 best X" article should yield up to 10 entities.
- For each entity, fill every requested column from the source text.
- If a value is explicitly stated, set state="filled" and copy the exact supporting text into evidence_snippet.
- If you looked and the value is not present, set state="not_found", value_text=null.
- If the column is irrelevant to this source, set state="unsupported", value_text=null.
- NEVER guess or infer values not grounded in the source text.
- Return a JSON array of entity objects."""

EXTRACTOR_SYSTEM_ENTITY = """\
You are extracting a single entity's data from its dedicated page (profile, product page, about page, or review).

Rules:
- Extract exactly ONE entity with maximum detail for every column.
- Every value_text must be directly supported by text on the page. Copy the supporting passage into evidence_snippet.
- If a value is not on this page, set state="not_found", value_text=null. Do not guess.
- If this page does not contain a specific entity matching the query (e.g. listicle, homepage, or aggregator), return an empty array [].
- Do NOT return the page title, site name, domain name, or URL as an entity name.
- Return a JSON array with one element."""

EXTRACTOR_SYSTEM_FORUM = """\
You are extracting entity mentions from a forum discussion or user-generated content page.

Rules:
- Only extract entities that are explicitly named and described with verifiable details.
- Set confidence lower (0.3-0.6) unless the post includes verifiable facts like URLs, addresses, or ratings.
- If no clear entities match the query, return an empty array [].
- Return a JSON array."""

EXTRACTOR_SYSTEM_BY_CLASS = {
    SourceClass.ROUNDUP:       EXTRACTOR_SYSTEM_ROUNDUP,
    SourceClass.DIRECTORY:     EXTRACTOR_SYSTEM_ROUNDUP,
    SourceClass.ENTITY_PAGE:   EXTRACTOR_SYSTEM_ENTITY,
    SourceClass.OFFICIAL_SITE: EXTRACTOR_SYSTEM_ENTITY,
    SourceClass.FORUM:         EXTRACTOR_SYSTEM_FORUM,
}

EXTRACTOR_USER = """\
Query: {query}
Entity type: {entity_type}
Source URL: {url}
Source title: {title}

Criteria to evaluate:
{criteria_json}

Columns to fill:
{columns_json}

Source text:
{body}

Return a JSON array:
[{{
  "canonical_name": "string",
  "canonical_url": "https://... or null",
  "extraction_confidence": 0.0,
  "cells": [
    {{"key": "col_key", "value_text": "string or null", "state": "filled|not_found|unsupported", "confidence": 0.0, "evidence_snippet": "exact quote from source"}}
  ],
  "criteria_verdicts": [
    {{"label": "criterion", "verdict": "pass|fail|uncertain", "summary": "one sentence", "confidence": 0.0, "evidence_snippet": "exact quote"}}
  ]
}}]"""


def extract_entities(
    query: str, entity_type: str,
    criteria: list[Criterion], columns: list[ColumnSpec],
    doc: ParsedDocument,
) -> list[ExtractedRow]:
    system = EXTRACTOR_SYSTEM_BY_CLASS.get(doc.source_class, EXTRACTOR_SYSTEM_ENTITY)
    user = EXTRACTOR_USER.format(
        query=query, entity_type=entity_type,
        url=doc.url, title=doc.title,
        criteria_json=json.dumps([{"label": c.label, "kind": c.kind} for c in criteria]),
        columns_json=json.dumps([{"key": c.key, "label": c.label, "kind": c.kind,
                                   "value_type": c.value_type} for c in columns]),
        body=doc.text[:12000],
    )

    # Route model: roundup/directory -> 3-flash (multi-entity reasoning),
    # entity_page/official/forum -> 3.1-flash-lite (simpler task, lower cost)
    op = ("extractor_roundup"
          if doc.source_class in (SourceClass.ROUNDUP, SourceClass.DIRECTORY)
          else "extractor_entity")
    result = gemini_call(op, system=system, user=user)
    if isinstance(result, dict):
        result = [result]
    if not isinstance(result, list):
        return []

    rows: list[ExtractedRow] = []
    for entity in result:
        cells = [
            CellValue(
                key=c.get("key", ""),
                value_text=c.get("value_text"),
                state=c.get("state", "not_found"),
                confidence=float(c.get("confidence", 0.3)),
                evidence_snippet=c.get("evidence_snippet", ""),
            )
            for c in entity.get("cells", [])
        ]
        verdicts = [
            CriterionVerdict(
                label=v.get("label", ""),
                verdict=v.get("verdict", "uncertain"),
                summary=v.get("summary", ""),
                confidence=float(v.get("confidence", 0.3)),
                evidence_snippet=v.get("evidence_snippet", ""),
            )
            for v in entity.get("criteria_verdicts", [])
        ]
        rows.append(ExtractedRow(
            canonical_name=entity.get("canonical_name", "Unknown"),
            canonical_url=entity.get("canonical_url") or "",
            cells=cells,
            criteria_verdicts=verdicts,
            extraction_confidence=float(entity.get("extraction_confidence", 0.5)),
            source_url=doc.url,
            source_class=doc.source_class,
        ))

    return rows


def is_junk(row: ExtractedRow, doc_url: str) -> bool:
    if row.extraction_confidence <= 0.0:
        return True
    if not row.canonical_name:
        return True
    name_lower = row.canonical_name.lower().strip()
    if name_lower in {"not_found", "unknown", "n/a", "none", ""}:
        return True
    if sum(1 for c in row.cells if c.state == "filled") == 0:
        return True
    domain = urlparse(doc_url).netloc.replace("www.", "").split(".")[0]
    if name_lower == domain.lower():
        return True
    return False


# ── Extract from all fetched documents ────────────────────────────
all_extracted: list[ExtractedRow] = []

for url, doc in document_store.items():
    print(f"Extracting from [{doc.source_class.value}] {doc.title[:60]}...")
    try:
        rows = extract_entities(query, plan.entity_type, plan.criteria, plan.columns, doc)
        kept_rows = [r for r in rows if not is_junk(r, doc.url)]
        all_extracted.extend(kept_rows)
        print(f"  -> {len(kept_rows)} entities")
        for r in kept_rows:
            filled = sum(1 for c in r.cells if c.state == "filled")
            print(f"     {r.canonical_name}  ({filled}/{len(r.cells)} filled, "
                  f"conf={r.extraction_confidence:.2f})")
    except Exception as e:
        print(f"  FAILED: {e}")
    print()

print(f"\nTotal extracted rows: {len(all_extracted)}")

Extracting from [roundup] The Best Whole-Pie Pizza Spots in Brooklyn Right Now...
  FAILED: Gemini failed on all models (extractor_roundup): HTTPSConnectionPool(host='generativelanguage.googleapis.com', port=443): Read timed out. (read timeout=30)

Extracting from [roundup] Dough Masters: The 16 Best Pizza Restaurants in Brooklyn | B...
    (used fallback gemini-2.5-flash for extractor_roundup)
  -> 4 entities
     Di Fara Pizza  (4/8 filled, conf=1.00)
     Luigi’s Pizza  (5/8 filled, conf=1.00)
     F&F Pizzeria  (4/8 filled, conf=1.00)
     Baby Luc’s  (4/8 filled, conf=1.00)

Extracting from [roundup] The 20 Best Pizza Places In Brooklyn...
    (used fallback gemini-2.5-flash for extractor_roundup)
  -> 5 entities
     L'Industrie Pizzeria  (7/8 filled, conf=1.00)
     Lucali  (6/8 filled, conf=1.00)
     L&B Spumoni Gardens  (8/8 filled, conf=1.00)
     Chrissy's Pizza  (7/8 filled, conf=1.00)
     Di Fara Pizza  (7/8 filled, conf=1.00)

Extracting from [roundup] 12 Best Pizza Joi

## Cell 5 — Fuzzy Entity Dedup and Merge

Normalized name exact match first, then Jaccard token overlap for fuzzy merge. Keeps highest-confidence cell value per column across all source documents.

In [8]:
def normalize_name(name: str) -> str:
    if not name:
        return ""
    name = name.lower().strip()
    name = re.sub(r"['\u2018\u2019\u2032`]", "", name)
    name = re.sub(r"[^\w\s]", " ", name)
    return re.sub(r"\s+", " ", name).strip()


def jaccard_tokens(a: str, b: str) -> float:
    ta = set(normalize_name(a).split())
    tb = set(normalize_name(b).split())
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


def dedupe_and_merge(rows: list[ExtractedRow], threshold: float = 0.7) -> list[MergedRow]:
    # Phase 1: exact normalized-name grouping
    groups: dict[str, list[ExtractedRow]] = {}
    for row in rows:
        groups.setdefault(normalize_name(row.canonical_name), []).append(row)

    # Phase 2: fuzzy merge groups above threshold
    keys = list(groups.keys())
    absorbed: set[str] = set()
    for i, ka in enumerate(keys):
        if ka in absorbed:
            continue
        for kb in keys[i + 1:]:
            if kb in absorbed:
                continue
            if jaccard_tokens(ka, kb) >= threshold:
                groups[ka].extend(groups[kb])
                absorbed.add(kb)
    for k in absorbed:
        del groups[k]

    # Phase 3: merge each group into one MergedRow
    result: list[MergedRow] = []
    for _, group in groups.items():
        best = max(group, key=lambda r: r.extraction_confidence)

        cells_by_key: dict[str, CellValue] = {}
        for row in group:
            for cell in row.cells:
                prev = cells_by_key.get(cell.key)
                if prev is None or (
                    cell.state == "filled"
                    and (prev.state != "filled" or cell.confidence > prev.confidence)
                ):
                    cells_by_key[cell.key] = cell

        verdicts_by_label: dict[str, CriterionVerdict] = {}
        for row in group:
            for v in row.criteria_verdicts:
                prev = verdicts_by_label.get(v.label)
                if prev is None or v.confidence > prev.confidence:
                    verdicts_by_label[v.label] = v

        avg_score = sum(r.extraction_confidence for r in group) / len(group)

        result.append(MergedRow(
            canonical_name=best.canonical_name,
            canonical_url=best.canonical_url or "",
            cells=list(cells_by_key.values()),
            criteria_verdicts=list(verdicts_by_label.values()),
            score=avg_score,
            source_urls=list({r.source_url for r in group}),
        ))

    return sorted(result, key=lambda r: r.score, reverse=True)


# ── Test ──────────────────────────────────────────────────────────
merged_rows = dedupe_and_merge(all_extracted)

print(f"Before dedup: {len(all_extracted)} rows")
print(f"After dedup:  {len(merged_rows)} rows\n")

for i, row in enumerate(merged_rows[:15]):
    filled = sum(1 for c in row.cells if c.state == "filled")
    print(f"  {i+1:2d}. {row.canonical_name}")
    print(f"      URL: {row.canonical_url}")
    print(f"      Cells: {filled}/{len(row.cells)} filled | "
          f"Score: {row.score:.2f} | Sources: {len(row.source_urls)}")
    print()

Before dedup: 19 rows
After dedup:  15 rows

   1. Di Fara Pizza
      URL: https://www.difarapizzany.com/
      Cells: 8/8 filled | Score: 1.00 | Sources: 2

   2. Luigi’s Pizza
      URL: http://luigispizzabrooklyn.com/
      Cells: 7/8 filled | Score: 1.00 | Sources: 2

   3. F&F Pizzeria
      URL: http://franks.pizza/
      Cells: 4/8 filled | Score: 1.00 | Sources: 1

   4. Baby Luc’s
      URL: http://www.babylucs.com/
      Cells: 4/8 filled | Score: 1.00 | Sources: 1

   5. L'Industrie Pizzeria
      URL: https://www.theinfatuation.com/new-york/reviews/lindustrie-pizzeria
      Cells: 8/8 filled | Score: 1.00 | Sources: 2

   6. Lucali
      URL: https://www.theinfatuation.com/new-york/reviews/lucali
      Cells: 6/8 filled | Score: 1.00 | Sources: 1

   7. Chrissy's Pizza
      URL: https://www.theinfatuation.com/new-york/reviews/chrissys-pizza-greenpoint
      Cells: 7/8 filled | Score: 1.00 | Sources: 1

   8. L’Industrie
      URL: https://www.lindustriebk.com/
      Cells

## Cell 6 — Supervisor ReAct Agent

Observes per-column fill rate + avg confidence, row count vs target, and available unfetched URLs. Decides one action per iteration: `search_more`, `fetch_more`, `extract_from_existing`, or `done`. Max 3 iterations.

In [9]:
def build_supervisor_state(
    plan: ResearchPlan,
    rows: list[MergedRow],
    search_results: list[SearchResult],
    iteration: int,
    target_rows: int,
    failed_urls: set[str] | None = None,
) -> SupervisorState:
    summaries: list[ColumnFillSummary] = []
    for col in plan.columns:
        filled = 0
        total_conf = 0.0
        for row in rows:
            cell = next((c for c in row.cells if c.key == col.key), None)
            if cell and cell.state == "filled":
                filled += 1
                total_conf += cell.confidence
        n = max(len(rows), 1)
        summaries.append(ColumnFillSummary(
            key=col.key, label=col.label,
            fill_rate=filled / n,
            avg_confidence=total_conf / max(filled, 1),
        ))

    fetched_urls = set(document_store.keys())
    blocked = failed_urls or set()
    unfetched = [
        r.url for r in search_results
        if r.url not in fetched_urls and r.url not in blocked
    ]

    return SupervisorState(
        total_rows=len(rows), target_rows=target_rows,
        column_summaries=summaries,
        unfetched_urls=unfetched[:10],
        iteration=iteration,
    )


SUPERVISOR_SYSTEM = """\
You are a research supervisor for a grounded entity discovery pipeline. After each iteration you observe the current state of the research table and decide ONE next action.

Actions:
- search_more: provide 2-3 new search queries to find more entities or fill data gaps. Use when row count is far below target or many columns are poorly filled.
- fetch_more: provide specific URLs from the unfetched list to fetch. Use when promising unfetched URLs exist.
- extract_from_existing: re-extract from an already-fetched document focusing on specific columns. Use when documents exist but columns were missed.
- done: the table is sufficiently complete. Use when row count is near target AND identity columns are well-filled.

Be decisive. Do not search_more if rows are already near target. Do not say done if identity columns are below 60% fill rate."""


def supervisor_decide(query: str, state: SupervisorState) -> SupervisorAction:
    col_table = "\n".join(
        f"  {s.label:20s}  fill={s.fill_rate:.0%}  avg_conf={s.avg_confidence:.2f}"
        for s in state.column_summaries
    )
    unfetched_list = "\n".join(f"  - {u}" for u in state.unfetched_urls[:5])

    user_msg = f"""\
Research query: {query}
Iteration: {state.iteration} of {state.max_iterations}
Rows found: {state.total_rows}  (target: {state.target_rows})

Column fill rates:
{col_table}

Unfetched URLs available: {len(state.unfetched_urls)}
{unfetched_list}

Return JSON:
{{
  "action": "search_more|fetch_more|extract_from_existing|done",
  "queries": ["..."],
  "urls": ["..."],
  "focus_columns": ["..."],
  "reasoning": "one sentence"
}}"""

    try:
        result = gemini_call("supervisor", system=SUPERVISOR_SYSTEM, user=user_msg)
    except RuntimeError as e:
        return SupervisorAction(
            action="done",
            reasoning=f"Supervisor skipped (API limit): {e}",
        )
    if not isinstance(result, dict):
        return SupervisorAction(
            action="done",
            reasoning=f"Supervisor got non-JSON response ({type(result).__name__})",
        )
    return SupervisorAction(
        action=result.get("action", "done"),
        queries=result.get("queries") or [],
        urls=result.get("urls") or [],
        focus_columns=result.get("focus_columns") or [],
        reasoning=result.get("reasoning", ""),
    )


# ── Test ──────────────────────────────────────────────────────────
state = build_supervisor_state(plan, merged_rows, all_search_results,
                                iteration=1, target_rows=target)
print("Supervisor state:")
print(f"  Rows: {state.total_rows}/{state.target_rows}")
print(f"  Column fill rates:")
for s in state.column_summaries:
    print(f"    {s.label:20s} {s.fill_rate:.0%}  (avg conf: {s.avg_confidence:.2f})")
print(f"  Unfetched URLs: {len(state.unfetched_urls)}")
print()

action = supervisor_decide(query, state)
print(f"Decision: {action.action}")
print(f"Reasoning: {action.reasoning}")
if action.queries:
    print(f"Queries: {action.queries}")
if action.urls:
    print(f"URLs: {action.urls}")
if action.focus_columns:
    print(f"Focus columns: {action.focus_columns}")

Supervisor state:
  Rows: 15/10
  Column fill rates:
    Name                 100%  (avg conf: 1.00)
    Website              80%  (avg conf: 1.00)
    Address              80%  (avg conf: 1.00)
    Neighborhood         80%  (avg conf: 0.97)
    Style (e.g., Neapolitan, NY Slice) 73%  (avg conf: 0.94)
    Price Range          53%  (avg conf: 1.00)
    Rating               47%  (avg conf: 1.00)
    Signature Pie        47%  (avg conf: 0.96)
  Unfetched URLs: 8

    (used fallback gemini-2.5-flash for supervisor)
Decision: fetch_more
Reasoning: Many columns have low fill rates, and promising unfetched URLs are available that are likely to contain the missing details for style, price, rating, and signature pies.
URLs: ['https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html', 'https://www.tastingtable.com/1432084/best-pizza-shops-brooklyn/', 'https://www.thedailymeal.com/1380010/ultimate-guide-best-pizza-places-brooklyn/', 'https://ny.eater.com/dining-report/407180/ops-

## Cell 7 — Query Rewriter

Gap-aware query generation: looks at poorly-filled columns and generates targeted search queries to fill the missing data.

In [10]:
REWRITER_SYSTEM = """\
You are a search query rewriter for a research pipeline. Given the original research query and columns that are poorly filled, generate 2-3 targeted web search queries designed to find the missing data.

Rules:
- Each query should target a specific data gap
- Include the entity type and location/context from the original query
- Avoid repeating the original search queries
- Return a JSON array of query strings"""


def rewrite_queries(
    original_query: str,
    column_gaps: list[ColumnFillSummary],
    entity_type: str,
) -> list[str]:
    gap_desc = "\n".join(
        f"  {g.label}: {g.fill_rate:.0%} filled, avg confidence {g.avg_confidence:.2f}"
        for g in column_gaps if g.fill_rate < 0.6
    )
    if not gap_desc:
        return []

    user_msg = f"""\
Original query: {original_query}
Entity type: {entity_type}

Poorly filled columns:
{gap_desc}

Generate 2-3 search queries to fill these gaps. Return a JSON array of strings."""

    result = gemini_call("rewriter", system=REWRITER_SYSTEM, user=user_msg)
    if isinstance(result, list):
        return [str(q) for q in result][:3]
    return []


# ── Test ──────────────────────────────────────────────────────────
gaps = [s for s in state.column_summaries if s.fill_rate < 0.6]
if gaps:
    new_queries = rewrite_queries(query, gaps, plan.entity_type)
    print("Rewritten queries for gap-filling:")
    for q in new_queries:
        print(f"  {q}")
else:
    print("No significant column gaps — skipping rewriter")

Rewritten queries for gap-filling:
  best pizza Brooklyn price range
  top rated pizza restaurants Brooklyn
  Brooklyn pizza places signature dishes


## Cell 8 — Verification

Rewritten verifier with system/user separation. No artificial budget cap — verifies all non-accepted rows.

In [14]:
VERIFIER_SYSTEM = """\
You are verifying extracted entity data from a grounded research pipeline. For each row, assess whether the extracted values are supported by the cited evidence.

Rules:
- Only change row_status if evidence clearly supports the change
- Prefer abstention over overclaiming — if evidence is weak, keep status as "uncertain"
- Check identity fields (name, URL) for accuracy
- Check that criteria verdicts align with evidence snippets
- Return your assessment as JSON"""


def verify_row(query: str, row: MergedRow, criteria: list[Criterion]) -> VerifiedRow:
    cells_desc = json.dumps([
        {"key": c.key, "value": c.value_text, "state": c.state,
         "confidence": c.confidence,
         "evidence": (c.evidence_snippet or "")[:200]}
        for c in row.cells
    ])
    verdicts_desc = json.dumps([
        {"label": v.label, "verdict": v.verdict, "summary": v.summary,
         "confidence": v.confidence}
        for v in row.criteria_verdicts
    ])

    user_msg = f"""\
Query: {query}
Entity: {row.canonical_name}
URL: {row.canonical_url}
Current status: {row.status}
Score: {row.score:.2f}
Sources: {', '.join(row.source_urls[:5])}

Cells: {cells_desc}
Criteria: {verdicts_desc}

Return JSON:
{{
  "row_status": "accepted|rejected|uncertain|conflict",
  "score": 0.0,
  "verification_summary": "one sentence",
  "criteria_verdicts": [
    {{"label": "string", "verdict": "pass|fail|uncertain", "summary": "string", "confidence": 0.0}}
  ]
}}"""

    result = gemini_call("verifier", system=VERIFIER_SYSTEM, user=user_msg)
    if not isinstance(result, dict):
        raise ValueError(f"Verifier returned {type(result).__name__}: {str(result)[:200]}")

    raw_verdicts = result.get("criteria_verdicts") or []
    verified_verdicts = [
        CriterionVerdict(
            label=v.get("label", ""), verdict=v.get("verdict", "uncertain"),
            summary=v.get("summary", ""), confidence=float(v.get("confidence", 0.3)),
        )
        for v in raw_verdicts
        if isinstance(v, dict)
    ]

    return VerifiedRow(
        canonical_name=row.canonical_name,
        canonical_url=row.canonical_url,
        cells=row.cells,
        criteria_verdicts=verified_verdicts or row.criteria_verdicts,
        score=float(result.get("score", row.score)),
        source_urls=row.source_urls,
        status=result.get("row_status", row.status),
        verification_summary=result.get("verification_summary", ""),
    )


# ── Verify all rows ──────────────────────────────────────────────
verified_rows: list[VerifiedRow] = []

for row in merged_rows:
    print(f"Verifying {row.canonical_name}...")
    try:
        v = verify_row(query, row, plan.criteria)
        verified_rows.append(v)
        print(f"  {row.status} -> {v.status}  (score: {v.score:.2f})")
        print(f"  {v.verification_summary}")
    except Exception as e:
        print(f"  FAILED: {e}")
        verified_rows.append(VerifiedRow(
            canonical_name=row.canonical_name, canonical_url=row.canonical_url,
            cells=row.cells, criteria_verdicts=row.criteria_verdicts,
            score=row.score, source_urls=row.source_urls, status=row.status,
        ))
    print()

status_counts: dict[str, int] = {}
for r in verified_rows:
    status_counts[r.status] = status_counts.get(r.status, 0) + 1
print(f"Verified {len(verified_rows)} rows")
print(f"Status distribution: {json.dumps(status_counts)}")

Verifying Di Fara Pizza...
  uncertain -> accepted  (score: 1.00)
  The entity is a well-documented, highly-rated pizza establishment in Brooklyn, and all extracted data points are supported by the provided evidence.

Verifying Luigi’s Pizza...
  uncertain -> accepted  (score: 1.00)
  The entity is well-supported by the provided evidence as a highly-rated pizza establishment in Brooklyn.

Verifying F&F Pizzeria...
  uncertain -> accepted  (score: 1.00)
  The entity is confirmed as a pizza restaurant in Brooklyn with sufficient evidence to support its inclusion in a 'best of' list despite subjective rating variations.

Verifying Baby Luc’s...
  uncertain -> accepted  (score: 1.00)
  The entity is confirmed as a pizza restaurant in Brooklyn with positive critical mention in the provided source.

Verifying L'Industrie Pizzeria...
  uncertain -> accepted  (score: 1.00)
  The entity is well-documented as a top-rated pizza establishment in Brooklyn, and all extracted fields are supported by 

In [13]:
merged_rows[1]

MergedRow(canonical_name='Luigi’s Pizza', canonical_url='http://luigispizzabrooklyn.com/', cells=[CellValue(key='name', value_text='Luigi’s Pizza', state='filled', confidence=1.0, evidence_snippet='### [Luigi’s Pizza](http://luigispizzabrooklyn.com/)'), CellValue(key='url', value_text='http://luigispizzabrooklyn.com/', state='filled', confidence=1.0, evidence_snippet='### [Luigi’s Pizza](http://luigispizzabrooklyn.com/)'), CellValue(key='address', value_text='686 5th Ave, Brooklyn, NY 11215, United States', state='filled', confidence=1.0, evidence_snippet="[686 5th Ave, Brooklyn, NY 11215, United States](https://www.google.com/maps/place/Luigi's+Pizza/@40.661655,-73.9959629,17z/data=!3m1!4b1!4m6!3m5!1s0x89c25ae614c7ff57:0xc2db102b7c600406!8m2!3d40.661655!4d-73.993388!16s%2Fg%2F1tfwl8k7?entry=ttu)"), CellValue(key='neighborhood', value_text='South Slope', state='filled', confidence=1.0, evidence_snippet='Luigi’s Pizza, South Slope'), CellValue(key='pizza_style', value_text='traditional 

## Cell 9 — Confidence-Weighted Ranking and CSV Export

Identity columns weighted 3x, enrichment 1x. Accepted rows boosted, rejected penalized. Final CSV includes source citations per row.

In [15]:
def compute_row_score(row, columns: list[ColumnSpec]) -> float:
    total_weight = 0.0
    weighted = 0.0
    for col in columns:
        w = 3.0 if col.kind == "identity" else 1.0
        total_weight += w
        cell = next((c for c in row.cells if c.key == col.key), None)
        if cell and cell.state == "filled":
            weighted += cell.confidence * w
    return weighted / max(total_weight, 1.0)


def rank_rows(rows: list[VerifiedRow], columns: list[ColumnSpec]) -> list[RankedRow]:
    scored: list[RankedRow] = []
    for row in rows:
        fs = compute_row_score(row, columns)
        if row.status == "accepted":
            fs *= 1.1
        elif row.status == "rejected":
            fs *= 0.3
        scored.append(RankedRow(
            canonical_name=row.canonical_name, canonical_url=row.canonical_url,
            cells=row.cells, criteria_verdicts=row.criteria_verdicts,
            score=row.score, source_urls=row.source_urls,
            status=row.status, verification_summary=row.verification_summary,
            final_score=min(fs, 1.0),
        ))
    scored.sort(key=lambda r: r.final_score, reverse=True)
    for i, r in enumerate(scored):
        r.rank = i + 1
    return scored


def export_csv(rows: list[RankedRow], columns: list[ColumnSpec]) -> str:
    buf = io.StringIO()
    w = csv.writer(buf)
    header = ["Rank", "Entity", "URL", "Status", "Score"]
    header += [c.label for c in columns]
    header += ["Sources"]
    w.writerow(header)

    for row in rows:
        by_key = {c.key: c for c in row.cells}
        vals = []
        for col in columns:
            cell = by_key.get(col.key)
            vals.append(cell.value_text if cell and cell.value_text else "—")
        w.writerow([
            row.rank, row.canonical_name, row.canonical_url,
            row.status, f"{row.final_score:.2f}",
            *vals, " | ".join(row.source_urls),
        ])
    return buf.getvalue()


# ── Test ──────────────────────────────────────────────────────────
ranked = rank_rows(verified_rows, plan.columns)

print(f"Final rankings ({len(ranked)} rows):\n")
for row in ranked[:15]:
    filled = sum(1 for c in row.cells if c.state == "filled")
    print(f"  #{row.rank:2d} [{row.status:10s}] {row.canonical_name}")
    print(f"      Score: {row.final_score:.2f} | Cells: {filled}/{len(row.cells)} filled")
    for cell in row.cells:
        if cell.state == "filled" and cell.value_text:
            print(f"      {cell.key}: {cell.value_text[:60]}")
    print()

csv_output = export_csv(ranked, plan.columns)
print("--- CSV Export (first 2000 chars) ---\n")
print(csv_output[:2000])

Final rankings (15 rows):

  # 1 [accepted  ] Di Fara Pizza
      Score: 1.00 | Cells: 8/8 filled
      name: Di Fara Pizza
      url: https://www.difarapizzany.com/
      address: 1424 Avenue J, Brooklyn, NY 11230, United States
      neighborhood: Midwood
      pizza_style: round slice, square slice
      price_range: $$$$
      rating: 9.3
      signature_pie: round slice, square slice

  # 2 [accepted  ] L'Industrie Pizzeria
      Score: 1.00 | Cells: 8/8 filled
      name: L'Industrie Pizzeria
      url: https://www.theinfatuation.com/new-york/reviews/lindustrie-p
      address: 254 S 2nd St, Brooklyn, NY 11211
      neighborhood: Williamsburg
      pizza_style: old school and new, with a thin but sturdy crust, charred in
      price_range: $$$$
      rating: 9.1
      signature_pie: burrata pie

  # 3 [accepted  ] Chrissy's Pizza
      Score: 1.00 | Cells: 7/8 filled
      name: Chrissy's Pizza
      url: https://www.theinfatuation.com/new-york/reviews/chrissys-piz
      address:

## Cell 10 — Full Pipeline with Supervisor Loop

Wires all cells into `run_pipeline(query, target_results)`. The supervisor loop runs up to 3 iterations of observe → decide → act before proceeding to verification and ranking.

Run on all 3 test queries to validate the refactored architecture.

In [16]:
def run_pipeline(query: str, target_results: int = 10) -> list[RankedRow]:
    global document_store, all_search_results
    document_store = {}
    all_search_results = []
    failed_urls: set[str] = set()

    print(f"\n{'=' * 70}")
    print(f"  PIPELINE: {query}")
    print(f"  Target: {target_results} results")
    print(f"{'=' * 70}\n")

    # ── Step 1: Plan ──────────────────────────────────────────────
    print("── Step 1: Planning ──")
    plan = plan_query(query, target_results)
    print(f"  Entity type: {plan.entity_type}")
    print(f"  Criteria: {len(plan.criteria)} | Columns: {len(plan.columns)} | "
          f"Queries: {len(plan.search_queries)}")

    # ── Step 2: Search ────────────────────────────────────────────
    print("\n── Step 2: Searching ──")
    all_search_results = search_brave(plan.search_queries, results_per_query=10)
    print(f"  {len(all_search_results)} unique URLs")

    # ── Step 3: Fetch + classify ──────────────────────────────────
    print("\n── Step 3: Fetching + classifying ──")
    fetch_limit = min(12, len(all_search_results))
    for r in all_search_results[:fetch_limit]:
        if r.url not in document_store and r.url not in failed_urls:
            doc = fetch_and_parse(r.url)
            if doc:
                doc.source_class = classify_source(doc, plan.entity_type)
                document_store[r.url] = doc
            else:
                failed_urls.add(r.url)
    class_counts = {}
    for d in document_store.values():
        class_counts[d.source_class.value] = class_counts.get(d.source_class.value, 0) + 1
    print(f"  {len(document_store)} documents: {json.dumps(class_counts)}")

    # ── Step 4: Extract ───────────────────────────────────────────
    print("\n── Step 4: Extracting entities ──")
    all_extracted: list[ExtractedRow] = []
    for url, doc in document_store.items():
        try:
            rows = extract_entities(query, plan.entity_type,
                                    plan.criteria, plan.columns, doc)
            kept_rows = [r for r in rows if not is_junk(r, doc.url)]
            all_extracted.extend(kept_rows)
            print(f"  [{doc.source_class.value:14s}] {doc.title[:50]} -> {len(kept_rows)} entities")
        except Exception as e:
            print(f"  FAILED: {doc.title[:50]} ({e})")
    print(f"  Total: {len(all_extracted)} extracted rows")

    # ── Step 5: Dedup ─────────────────────────────────────────────
    print("\n── Step 5: Dedup ──")
    merged = dedupe_and_merge(all_extracted)
    print(f"  {len(all_extracted)} -> {len(merged)} unique entities")

    # ── Step 6: Supervisor loop ───────────────────────────────────
    for iteration in range(1, 4):
        print(f"\n── Step 6: Supervisor (iteration {iteration}/3) ──")
        sup_state = build_supervisor_state(
            plan,
            merged,
            all_search_results,
            iteration,
            target_results,
            failed_urls=failed_urls,
        )
        action = supervisor_decide(query, sup_state)
        print(f"  Decision: {action.action}")
        print(f"  Reasoning: {action.reasoning}")

        if action.action == "done":
            break

        if action.action == "search_more":
            new_qs = action.queries
            if not new_qs:
                new_qs = rewrite_queries(query, sup_state.column_summaries,
                                         plan.entity_type)
            if new_qs:
                print(f"  Searching: {new_qs}")
                new_results = search_brave(new_qs, results_per_query=8)
                all_search_results.extend(new_results)
                for r in new_results[:6]:
                    if r.url not in document_store and r.url not in failed_urls:
                        doc = fetch_and_parse(r.url)
                        if doc:
                            doc.source_class = classify_source(doc, plan.entity_type)
                            document_store[r.url] = doc
                            try:
                                rows = extract_entities(
                                    query, plan.entity_type,
                                    plan.criteria, plan.columns, doc)
                                kept_rows = [x for x in rows if not is_junk(x, doc.url)]
                                all_extracted.extend(kept_rows)
                            except Exception as e:
                                print(f"  FAILED: {doc.title[:50]} ({e})")
                        else:
                            failed_urls.add(r.url)
                merged = dedupe_and_merge(all_extracted)
                print(f"  After iteration: {len(merged)} entities")

        elif action.action == "fetch_more":
            urls = action.urls or [
                r.url for r in all_search_results
                if r.url not in document_store and r.url not in failed_urls
            ][:4]
            for url in urls:
                if url in failed_urls:
                    continue
                if url not in document_store:
                    doc = fetch_and_parse(url)
                    if doc:
                        doc.source_class = classify_source(doc, plan.entity_type)
                        document_store[url] = doc
                        try:
                            rows = extract_entities(
                                query, plan.entity_type,
                                plan.criteria, plan.columns, doc)
                            kept_rows = [x for x in rows if not is_junk(x, doc.url)]
                            all_extracted.extend(kept_rows)
                        except Exception as e:
                            print(f"  FAILED: {doc.title[:50]} ({e})")
                    else:
                        failed_urls.add(url)
            merged = dedupe_and_merge(all_extracted)
            print(f"  After iteration: {len(merged)} entities")

        elif action.action == "extract_from_existing":
            for url, doc in list(document_store.items())[:3]:
                try:
                    rows = extract_entities(
                        query, plan.entity_type,
                        plan.criteria, plan.columns, doc)
                    kept_rows = [x for x in rows if not is_junk(x, doc.url)]
                    all_extracted.extend(kept_rows)
                except Exception as e:
                    print(f"  FAILED: {doc.title[:50]} ({e})")
            merged = dedupe_and_merge(all_extracted)
            print(f"  After iteration: {len(merged)} entities")

    # ── Step 7: Verify ────────────────────────────────────────────
    print("\n── Step 7: Verification ──")
    verified: list[VerifiedRow] = []
    for row in merged:
        try:
            v = verify_row(query, row, plan.criteria)
            verified.append(v)
        except:
            verified.append(VerifiedRow(
                canonical_name=row.canonical_name, canonical_url=row.canonical_url,
                cells=row.cells, criteria_verdicts=row.criteria_verdicts,
                score=row.score, source_urls=row.source_urls, status=row.status,
            ))
    sc = {}
    for r in verified:
        sc[r.status] = sc.get(r.status, 0) + 1
    print(f"  {json.dumps(sc)}")

    # ── Step 8: Rank + export ─────────────────────────────────────
    print("\n── Step 8: Ranking + export ──")
    ranked = rank_rows(verified, plan.columns)

    print(f"\n  Final table ({len(ranked)} rows):\n")
    for row in ranked[:15]:
        filled = sum(1 for c in row.cells if c.state == "filled")
        print(f"    #{row.rank:2d} [{row.status:10s}] {row.canonical_name}")
        print(f"        Score: {row.final_score:.2f} | "
              f"Cells: {filled}/{len(row.cells)} | Sources: {len(row.source_urls)}")

    csv_out = export_csv(ranked, plan.columns)
    print(f"\n  CSV: {len(csv_out)} chars")
    print(f"\n{'=' * 70}\n")

    return ranked


# ── Run on first test query ───────────────────────────────────────
results_1 = run_pipeline(*TEST_QUERIES[0])


  PIPELINE: best pizza places in Brooklyn
  Target: 10 results

── Step 1: Planning ──
  Entity type: business
  Criteria: 3 | Columns: 8 | Queries: 5

── Step 2: Searching ──
  24 unique URLs

── Step 3: Fetching + classifying ──
    Raw failed: https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html  (403 Client Error: Forbidden for url: https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html)
  Failed: https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html  (all fetch strategies failed)
    Raw failed: https://www.yelp.com/search?cflt=pizza&find_loc=Brooklyn,+NY  (403 Client Error: Forbidden for url: https://www.yelp.com/search?cflt=pizza&find_loc=Brooklyn,+NY)
  Failed: https://www.yelp.com/search?cflt=pizza&find_loc=Brooklyn,+NY  (all fetch strategies failed)
  10 documents: {"roundup": 9, "forum": 1}

── Step 4: Extracting entities ──
  FAILED: The Best Whole-Pie Pizza Spots in Brooklyn Right N (Gemini failed on all models (

RuntimeError: Gemini failed on all models (supervisor): 429 on gemini-2.5-flash

In [17]:
# ── Run on second test query ──────────────────────────────────────
results_2 = run_pipeline(*TEST_QUERIES[1])


  PIPELINE: YC W24 healthcare startups
  Target: 15 results

── Step 1: Planning ──
  Entity type: startup
  Criteria: 2 | Columns: 6 | Queries: 4

── Step 2: Searching ──
  21 unique URLs

── Step 3: Fetching + classifying ──
  12 documents: {"entity_page": 6, "roundup": 6}

── Step 4: Extracting entities ──
  [entity_page   ] Healthcare Startups funded by Y Combinator (YC) 20 -> 1 entities
  [entity_page   ] Healthcare Services Startups funded by Y Combinato -> 1 entities
  [entity_page   ] Healthcare IT Startups funded by Y Combinator (YC) -> 1 entities
  FAILED: Healthcare Startups funded by Y Combinator (YC) in (Gemini failed on all models (extractor_roundup): 429 on gemini-2.5-flash)
  [entity_page   ] The YC Startup Directory | Y Combinator -> 0 entities
  FAILED: Healthcare IT Startups funded by Y Combinator (YC) (Gemini failed on all models (extractor_roundup): 429 on gemini-2.5-flash)
  [entity_page   ] Health Tech Startups funded by Y Combinator (YC) 2 -> 1 entities
  FAILE

RuntimeError: Gemini failed on all models (supervisor): 429 on gemini-2.5-flash

In [13]:
# ── Run on third test query ───────────────────────────────────────
results_3 = run_pipeline(*TEST_QUERIES[2])


  PIPELINE: open source LLM projects with >1k stars
  Target: 12 results

── Step 1: Planning ──
  Entity type: software_project
  Criteria: 3 | Columns: 7 | Queries: 5

── Step 2: Searching ──
  30 unique URLs

── Step 3: Fetching + classifying ──
  12 documents: {"entity_page": 10, "forum": 2}

── Step 4: Extracting entities ──
  [entity_page   ] https://github.com/Hannibal046/Awesome-LLM -> 1 entities
  [entity_page   ] https://github.com/alvinreal/awesome-opensource-ai -> 1 entities
  [forum         ] Reddit - Please wait for verification -> 0 entities
  [entity_page   ] https://github.com/eugeneyan/open-llms -> 1 entities
  [entity_page   ] https://github.com/jihoo-kim/awesome-production-ll -> 1 entities
  [forum         ] Reddit - Please wait for verification -> 0 entities
  [entity_page   ] https://github.com/Shubhamsaboo/awesome-llm-apps -> 1 entities
  [entity_page   ] llm · GitHub Topics · GitHub -> 1 entities
  [entity_page   ] https://github.com/vince-lam/awesome-local-llm

AttributeError: 'NoneType' object has no attribute 'lower'

In [14]:
# ── Summary across all runs ───────────────────────────────────────
print("=" * 70)
print("  SUMMARY")
print("=" * 70)
for (q, t), results in zip(TEST_QUERIES, [results_1, results_2, results_3]):
    accepted = sum(1 for r in results if r.status == "accepted")
    avg_score = sum(r.final_score for r in results) / max(len(results), 1)
    avg_cells = sum(
        sum(1 for c in r.cells if c.state == "filled") / max(len(r.cells), 1)
        for r in results
    ) / max(len(results), 1)
    print(f"\n  Query: {q}")
    print(f"    Rows: {len(results)} (target: {t})")
    print(f"    Accepted: {accepted}")
    print(f"    Avg score: {avg_score:.2f}")
    print(f"    Avg cell fill: {avg_cells:.0%}")

  SUMMARY


NameError: name 'results_3' is not defined

## Learnings — What Broke and Why

### 1. Verifier NoneType crash
Gemini preview models sometimes returned JSON `null` instead of a valid object. Every downstream `.get()` call crashed. Fix: null guard in shared `gemini_call` and type guard in `verify_row`.

### 2. Apostrophe dedup failure
LLMs emitted different apostrophe variants (`'` vs `’`) for the same entity name. The normalizer handled them differently, creating phantom duplicates. Fix: strip all quote-like characters before tokenization.

### 3. Source misclassification
Title-based roundup detection with strict digit-first patterns missed spelled-out and flexible listicle titles (for example `Nine Best`, `Best Slice Shops`, and `19 Absolute Best`). Fix: broader regex and expanded roundup domain allowlist.

### 4. Fetcher blocked by bot protection
Raw HTTP requests were blocked (403) by TripAdvisor/Yelp and got interstitial content on Reddit. Fix: Jina Reader as primary fetcher, Reddit `.json` API route, and raw HTTP as fallback.

### 5. Garbage entity extraction
Entity-page extraction sometimes returned page titles or site names as entities when no entity was present. Fix: explicit extractor refusal rule (`return []` for non-entity pages) and a post-extraction quality gate.

### 6. Supervisor retry loop on dead URLs
The supervisor repeatedly selected known blocked URLs. Fix: maintain `failed_urls` and exclude them from the supervisor state and fetch queue.

### 7. None canonical_name crash
LLM extraction can return `canonical_name: null`. `normalize_name` called `.lower()` on `None` and crashed dedup. Fix: early guard for empty names.